# Compute

In [ ]:
import os
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov
import logging
from datetime import datetime
from joblib import Parallel, delayed

# Setup logging configuration
def setup_logger(sample_name=None):
    """Configure the logger"""
    logger = logging.getLogger('MetaCellPipeline')
    logger.setLevel(logging.DEBUG)
    
    # Clear previous handlers to avoid duplicate logs
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)
    
    # Console output format
    console_handler = logging.StreamHandler()
    console_format = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(console_format)
    
    # File output format (including sample name)
    os.makedirs('./process/logs/', exist_ok=True)
    log_file = f'./process/logs/pipeline_{sample_name if sample_name else "global"}.log'
    file_handler = logging.FileHandler(log_file, mode='w')
    file_format = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(file_format)
    
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger

# Initialize global logger
global_logger = setup_logger()
global_logger.info("Starting MetaCell processing pipeline")

# Data Loading
try:
    global_logger.info("Loading RNA data...")
    rna = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/adata.h5ad')
    global_logger.info(f"RNA data loaded: {rna.shape[0]} cells, {rna.shape[1]} features")
    
    global_logger.info("Loading ATAC data...")
    atac = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/atac/data_new.h5ad')
    global_logger.info(f"ATAC data loaded: {atac.shape[0]} cells, {atac.shape[1]} features")
except Exception as e:
    global_logger.error(f"Data loading failed: {str(e)}")
    raise

# Merge Data
try:
    global_logger.info("Merging RNA and ATAC data...")
    adata = sc.concat([rna, atac], axis=1)
    adata.layers['counts'] = adata.X
    adata.obs = rna.obs
    del rna
    del atac
    global_logger.info(f"Merged data shape: {adata.shape[0]} cells, {adata.shape[1]} features")
except Exception as e:
    global_logger.error(f"Data merging failed: {str(e)}")
    raise

# Add Dimensionality Reduction Results and Annotations
try:
    global_logger.info("Adding spectral embedding...")
    spectral = np.load('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
    adata.obsm['X_spectral'] = spectral
    global_logger.info(f"Spectral embedding added: {spectral.shape}")
    
    global_logger.info("Adding cell type annotations...")
    ann = pd.read_csv('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
    adata.obs['l3'] = ann.loc[adata.obs_names, 'l3'].values
    global_logger.info(f"Added {adata.obs['l3'].nunique()} unique cell types")
except Exception as e:
    global_logger.error(f"Annotation addition failed: {str(e)}")
    raise

# Create Output Directories
os.makedirs('./process/model/', exist_ok=True)
os.makedirs('./process/ad/', exist_ok=True)
os.makedirs('./process/cell_assign/', exist_ok=True)
os.makedirs('./process/logs/', exist_ok=True)

# Sample Processing Function
def process_sample(sample):
    """Complete pipeline for processing a single sample"""
    sample_logger = setup_logger(sample_name=sample)
    sample_logger.info(f"Starting processing for sample: {sample}")
    
    try:
        # 1. Create data subset
        sample_logger.info("Creating subset...")
        subset = adata[adata.obs['sample'] == sample]
        sample_logger.info(f"Subset created: {subset.shape[0]} cells")
        
        # 2. Initialize MetaCell
        sample_logger.info("Initializing MetaCell...")
        meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu'
        )
        
        # 3. Train model
        sample_logger.info("Training MetaCell model...")
        meta_obj.initialize_archetypes()
        meta_obj.train(min_iter=10, max_iter=50)
        sample_logger.info("Model training completed")
        
        # 4. Save model
        model_path = os.path.join('./process/model/', f"{sample}.pkl")
        meta_obj.save(model_path)
        sample_logger.info(f"Model saved to {model_path}")
        
        # 5. Predict metacells
        sample_logger.info("Predicting metacells...")
        ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')
        
        # 6. Calculate metrics
        sample_logger.info("Calculating separation and compactness...")
        ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
        ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()
        
        # 7. Save results
        ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
        ad.write_h5ad(ad_path)
        sample_logger.info(f"Anndata saved to {ad_path}")
        
        assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
        meta_obj.adata.obs[['SEACell']].to_csv(assign_path)
        sample_logger.info(f"Cell assignments saved to {assign_path}")
        
        # 8. Clean up resources
        torch.cuda.empty_cache()
        sample_logger.info(f"Completed processing for sample: {sample}")
        
        return True
    except Exception as e:
        sample_logger.error(f"Error processing sample {sample}: {str(e)}", exc_info=True)
        return False

# Main Execution Flow
if __name__ == '__main__':
    try:
        global_logger.info("Starting sample processing...")
        sample_counts = pd.DataFrame(adata.obs['sample'].value_counts().sort_values(ascending=True))
        global_logger.info(f"Found {len(sample_counts)} samples to process")
        
        # Parallel processing (adjust n_jobs based on server cores)
        results = Parallel(n_jobs=10)(
            delayed(process_sample)(sample) 
            for sample in sample_counts.index
        )
        
        # Calculate statistics
        success_count = sum(results)
        global_logger.info(
            f"Processing completed. Success: {success_count}/{len(sample_counts)} "
            f"({success_count/len(sample_counts):.1%})"
        )
        
        # Generate summary report
        report_path = './process/logs/summary_report.txt'
        with open(report_path, 'w') as f:
            f.write(f"MetaCell Processing Summary\n{'='*30}\n")
            f.write(f"Timestamp: {datetime.now()}\n")
            f.write(f"Total samples: {len(sample_counts)}\n")
            f.write(f"Successful: {success_count}\n")
            f.write(f"Failed: {len(sample_counts) - success_count}\n")
            f.write("\nSample Details:\n")
            for sample, status in zip(sample_counts.index, results):
                f.write(f"{sample}: {'SUCCESS' if status else 'FAILED'}\n")
        
        global_logger.info(f"Summary report saved to {report_path}")
        
    except Exception as e:
        global_logger.error(f"Fatal error in main pipeline: {str(e)}", exc_info=True)
        raise

## 'Cheong_Cell_2023_1-jcov124' The number of cells is 665, an error is reported, and it has been solved

In [ ]:
import os
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov
import logging
from datetime import datetime
from joblib import Parallel, delayed

In [2]:
rna = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/adata.h5ad')

In [3]:
rna

AnnData object with n_obs × n_vars = 1641866 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann'

In [4]:
atac = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/atac/data_new.h5ad')

In [5]:
import numpy as np
from scipy import sparse

def calculate_sparsity(adata):
    if sparse.issparse(adata.X):
        sparsity = 1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    else:
        sparsity = np.mean(adata.X == 0)
    return sparsity

sparsity = calculate_sparsity(rna)
print(f"Sparsity: {sparsity:.2%}")  

Sparsity: 95.06%


In [6]:
sparsity = calculate_sparsity(atac)
print(f"Sparsity: {sparsity:.2%}")  

Sparsity: 98.87%


In [ ]:
adata = sc.concat([rna, atac], axis=1)

In [ ]:
adata.layers['counts'] = adata.X
adata.obs = rna.obs

In [ ]:
rna = adata

In [ ]:
del adata
del atac

In [5]:
spectral = np.load('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
rna.obsm['X_spectral'] = spectral
    
ann = pd.read_csv('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
rna.obs['l3'] = ann.loc[rna.obs_names, 'l3'].values

In [6]:
sample_counts = pd.DataFrame(rna.obs['sample'].value_counts().sort_values(ascending=True))

In [7]:
sample_counts

,count
sample,
Cheong_Cell_2023_1-jcov124,665
Cheong_Cell_2023_1-HDu,1499
Marderstein_Nature_2024-TS21_4_H,1622
NeurIPS_2021-site3_donor6,1679
NeurIPS_2021-site3_donor7,1771
...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00599_02,27541
Thomson_Nat_Immunol_2024-B076_P1_PB00368_04,27766
Thomson_Nat_Immunol_2024-B076_P1_PB00124_02,28942


In [8]:
sample = 'Cheong_Cell_2023_1-jcov124'

In [9]:
subset = rna[rna.obs['sample'] == sample]

In [10]:
subset

View of AnnData object with n_obs × n_vars = 665 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann', 'l3'
    obsm: 'X_spectral'
    layers: 'counts'

In [11]:
subset.obs['l3'].value_counts()

l3
Tcm/Naive helper T cells       140
Type 1 helper T cells          130
Classical monocytes            111
HSC/MPP                         92
CD16+ NK cells                  61
Tem/Trm cytotoxic T cells       59
Memory B cells                  13
ELP                             11
Tem/Temra cytotoxic T cells      9
Naive B cells                    7
Non-classical monocytes          6
Mono-mac                         5
DC3                              5
Follicular helper T cells        4
Regulatory T cells               4
Transitional NK                  3
Mast cells                       1
ILC3                             1
MEMP                             1
pDC                              1
CMP                              1
Name: count, dtype: int64

In [12]:
meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu',
            n_metacells=9  
        )

Welcome to SEACells GPU!


In [13]:
meta_obj.initialize_archetypes()
meta_obj.train(min_iter=10, max_iter=50)

Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/665 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/665 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/665 [00:00<?, ?it/s]

Constructing CSR matrix...
Building kernel on X_spectral
Computing diffusion components from X_spectral for waypoint initialization ... 
Done.
Sampling waypoints ...
Done.
Selecting 9 cells from waypoint initialization.
Initializing residual matrix using greedy column selection
Initializing f and g...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 2744.07it/s]

Selecting 0 cells from greedy initialization.
Randomly initialized A matrix.


Setting convergence threshold at 0.04622
Starting iteration 1.
Completed iteration 1.
Converged after 9 iterations.
Starting iteration 10.
Completed iteration 10.


In [14]:
model_path = os.path.join('./process/model/', f"{sample}.pkl")
meta_obj.save(model_path)

In [15]:
ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:01<00:00,  8.27it/s]


In [16]:
ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()

In [27]:
ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
ad.write_h5ad(ad_path)

In [28]:
assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
meta_obj.adata.obs[['SEACell']].to_csv(assign_path)

## 'Thomson_Nat_Immunol_2024-B065_P1_PB00593_04' reported an error when calculating separation, which has been solved

In [27]:
sample = 'Thomson_Nat_Immunol_2024-B065_P1_PB00593_04'

In [41]:
subset = rna[rna.obs['sample'] == sample]

In [42]:
meta_obj = ov.single.MetaCell(
            subset,
            use_rep='X_spectral',
            use_gpu='cuda' if torch.cuda.is_available() else 'cpu'
        )

Welcome to SEACells GPU!


In [43]:
model_path = os.path.join('./process/model/', f"{sample}.pkl")
meta_obj.load(model_path)

In [44]:
ad = meta_obj.predicted(method='soft', celltype_label='l3', summarize_layer='counts')

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [07:14<00:00,  2.91s/it]


In [45]:
ad.obs['separation'] = meta_obj.separation(use_rep='X_spectral', nth_nbr=1)['separation'].tolist()
ad.obs['compactness'] = meta_obj.compactness(use_rep='X_spectral')['compactness'].tolist()

In [46]:
ad_path = os.path.join('./process/ad/', f"{sample}.h5ad")
ad.write_h5ad(ad_path)

In [47]:
assign_path = os.path.join('./process/cell_assign/', f"{sample}.csv")
meta_obj.adata.obs[['SEACell']].to_csv(assign_path)

# Merge and analysis

## h5ad concat

In [5]:
import os
import scanpy as sc
import numpy as np
import pandas as pd

adata_list = []
folder_path  = './process/ad/'

In [6]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".h5ad"):
        base_name = file_name.replace(".h5ad", "")
        print(f"处理文件: {file_name}")
        adata = sc.read_h5ad(os.path.join(folder_path, file_name))
        adata.obs_names = base_name + '@' + adata.obs_names
        adata_list.append(adata)

处理文件: Cheong_Cell_2023_1-HDu.h5ad
处理文件: NeurIPS_2021-site3_donor6.h5ad
处理文件: Wimmers_Cell_2023-44_CC0111.h5ad
处理文件: Marderstein_Nature_2024-TS21_4_H.h5ad
处理文件: Boukhaled_Nat_Immunol_2022-P5.h5ad
处理文件: NeurIPS_2021-site3_donor7.h5ad
处理文件: Wimmers_Cell_2023-76_CC0829.h5ad
处理文件: Wimmers_Cell_2023-40_CC0069.h5ad
处理文件: Marderstein_Nature_2024-H_5_B.h5ad
处理文件: Oguchi_Science_2024-D1_CD4bulk_2.h5ad
处理文件: Marderstein_Nature_2024-TS21_4_I.h5ad
处理文件: Oguchi_Science_2024-D1_Bulk_aCD4_1.h5ad
处理文件: 10x_datasets_9-1.h5ad
处理文件: Marderstein_Nature_2024-H_5_A.h5ad
处理文件: 10x_datasets_3-1.h5ad
处理文件: Cheong_Cell_2023_1-jcov21.h5ad
处理文件: Marderstein_Nature_2024-H_7_B.h5ad
处理文件: 10x_datasets_7-1.h5ad
处理文件: 10x_datasets_5-1.h5ad
处理文件: Cheong_Cell_2023_1-lgtd18.h5ad
处理文件: Wimmers_Cell_2023-72_CC0688.h5ad
处理文件: Thomson_Nat_Immunol_2024-B069_P1_IMM19_438.h5ad
处理文件: Thomson_Nat_Immunol_2024-B065_P1_IMM19_435.h5ad
处理文件: Cheong_Cell_2023_1-jcov52.h5ad
处理文件: Wimmers_Cell_2023-59_CC0256.h5ad
处理文件: Wimmers_Cell_2023-

In [7]:
merged_adata = sc.concat(adata_list, join="outer")
merged_adata

AnnData object with n_obs × n_vars = 21795 × 564060
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [8]:
merged_adata.write('./process/adata.h5ad')

### split rna

In [9]:
combined_genes = merged_adata.var_names[:36601].tolist() + \
                 merged_adata.var_names[[564058, 564059]].tolist()

In [19]:
rna = merged_adata[:,:36601]
rna

View of AnnData object with n_obs × n_vars = 21795 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [20]:
import numpy as np
from scipy import sparse

def calculate_sparsity(adata):
    if sparse.issparse(adata.X):
        sparsity = 1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1])
    else:
        sparsity = np.mean(adata.X == 0)
    return sparsity

sparsity = calculate_sparsity(rna)
print(f"Sparsity: {sparsity:.2%}") 

Sparsity: 57.57%


In [39]:
rna.write('./process/rna.h5ad')

### split atac

In [30]:
merged_adata

AnnData object with n_obs × n_vars = 21795 × 564060
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [41]:
combined_peaks = merged_adata.var_names[36599:564058].tolist()

In [45]:
len(combined_peaks), combined_peaks[527459-1]

(527459, 'chrX-9997483-9997983')

In [47]:
atac = merged_adata[:, combined_peaks]
atac

View of AnnData object with n_obs × n_vars = 21795 × 527459
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [48]:
sparsity = calculate_sparsity(atac)
print(f"Sparsity: {sparsity:.2%}")  # 例如 "Sparsity: 95.23%"

Sparsity: 69.12%


In [49]:
atac.write('./process/atac.h5ad')

In [54]:
merged_adata.obs

,Pseudo-sizes,celltype,celltype_purity,separation,compactness
Cheong_Cell_2023_1-HDu@0,63.736985,Tem/Trm cytotoxic T cells,0.480005,23.204641,0.009738
Cheong_Cell_2023_1-HDu@1,65.422619,Tcm/Naive helper T cells,0.468285,12.636931,30.667777
Cheong_Cell_2023_1-HDu@2,77.541340,Naive B cells,0.503537,0.101567,0.010034
Cheong_Cell_2023_1-HDu@3,72.350554,Tcm/Naive helper T cells,0.960469,0.101567,0.018626
Cheong_Cell_2023_1-HDu@4,64.618502,CD16+ NK cells,0.830080,1.198503,0.009845
...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,114.700387,Tcm/Naive helper T cells,0.994548,0.045278,0.012154
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,110.842287,Tcm/Naive helper T cells,1.000000,0.046566,0.002396
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,53.609057,Regulatory T cells,0.855926,0.116286,0.034826
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,50.190145,Type 1 helper T cells,0.981109,0.058782,0.016925


## cell_assign

In [6]:
import os

csv_list = []
folder_path  = './process/cell_assign'

In [7]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        base_name = file_name.replace(".csv", "")
        print(f"处理文件: {file_name}")
        data = pd.read_csv(os.path.join(folder_path, file_name), index_col=0)
        data['SEACell'] = base_name+'@'+data['SEACell'].str.replace('SEACell-', '')
        csv_list.append(data)

处理文件: Cheong_Cell_2023_1-HDu.csv
处理文件: NeurIPS_2021-site3_donor6.csv
处理文件: Wimmers_Cell_2023-44_CC0111.csv
处理文件: Marderstein_Nature_2024-TS21_4_H.csv
处理文件: Boukhaled_Nat_Immunol_2022-P5.csv
处理文件: NeurIPS_2021-site3_donor7.csv
处理文件: Wimmers_Cell_2023-76_CC0829.csv
处理文件: Wimmers_Cell_2023-40_CC0069.csv
处理文件: Marderstein_Nature_2024-H_5_B.csv
处理文件: Oguchi_Science_2024-D1_CD4bulk_2.csv
处理文件: Marderstein_Nature_2024-TS21_4_I.csv
处理文件: Oguchi_Science_2024-D1_Bulk_aCD4_1.csv
处理文件: 10x_datasets_9-1.csv
处理文件: Marderstein_Nature_2024-H_5_A.csv
处理文件: 10x_datasets_3-1.csv
处理文件: Cheong_Cell_2023_1-jcov21.csv
处理文件: Marderstein_Nature_2024-H_7_B.csv
处理文件: 10x_datasets_7-1.csv
处理文件: 10x_datasets_5-1.csv
处理文件: Cheong_Cell_2023_1-lgtd18.csv
处理文件: Wimmers_Cell_2023-72_CC0688.csv
处理文件: Thomson_Nat_Immunol_2024-B069_P1_IMM19_438.csv
处理文件: Thomson_Nat_Immunol_2024-B065_P1_IMM19_435.csv
处理文件: Cheong_Cell_2023_1-jcov52.csv
处理文件: Wimmers_Cell_2023-59_CC0256.csv
处理文件: Wimmers_Cell_2023-41_CC0107.csv
处理文件: Cheon

In [8]:
merged_df = pd.concat(csv_list, axis=0)
merged_df

,SEACell
Cheong_Cell_2023_1-AAACAGCCACCGTTCC-HDu,Cheong_Cell_2023_1-HDu@10
Cheong_Cell_2023_1-AAACAGCCATGTGGGA-HDu,Cheong_Cell_2023_1-HDu@4
Cheong_Cell_2023_1-AAACCGGCACCGGTAT-HDu,Cheong_Cell_2023_1-HDu@6
Cheong_Cell_2023_1-AAACGCGCAGCTAACC-HDu,Cheong_Cell_2023_1-HDu@12
Cheong_Cell_2023_1-AAACGTACAAGGCCAA-HDu,Cheong_Cell_2023_1-HDu@18
...,...
Thomson_Nat_Immunol_2024-5d46141253bc11edbeb90242ac110002-B076_P1_PB00353_03,Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@176
Thomson_Nat_Immunol_2024-5374991853bc11ed8ce30242ac110002-B076_P1_PB00353_03,Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@242
Thomson_Nat_Immunol_2024-5374a9c653bc11ed8ce30242ac110002-B076_P1_PB00353_03,Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@118
Thomson_Nat_Immunol_2024-60e8bd5e53bc11eda4150242ac110002-B076_P1_PB00353_03,Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@125


## umap

In [9]:
adata = rna

In [10]:
merged_df = merged_df.reindex(adata.obs_names)
merged_df

,SEACell
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-lane1@12
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-lane1@14
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-lane1@15
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-lane1@52
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-lane1@42
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Marderstein_Nature_2024-H_7_C@23
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Marderstein_Nature_2024-H_7_C@6
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Marderstein_Nature_2024-H_7_C@8
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Marderstein_Nature_2024-H_7_C@32


In [15]:
adata.obs['SEACell'] = merged_df['SEACell']

In [12]:
umap = np.load('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
umap

array([[14.6315975 ,  7.1658854 ],
       [15.238118  ,  8.004213  ],
       [11.487208  ,  6.272271  ],
       ...,
       [-0.74915725,  6.5377336 ],
       [-0.9178162 ,  7.9699144 ],
       [ 0.06110927,  8.532203  ]], dtype=float32)

In [13]:
adata.obsm['X_umap'] = umap

In [16]:
umap_coords = adata.obsm['X_umap']  # shape: (n_cells, 2)
metacell_labels = adata.obs[['SEACell']]  # metacell

df = pd.DataFrame(umap_coords, columns=['UMAP_1', 'UMAP_2'])
df['metacell'] = metacell_labels['SEACell'].tolist()

metacell_coords = df.groupby('metacell')[['UMAP_1', 'UMAP_2']].mean()

print(metacell_coords)

                                 UMAP_1     UMAP_2
metacell                                          
10x_datasets_1-1@0             9.074406  10.579405
10x_datasets_1-1@1            -3.235354   3.318362
10x_datasets_1-1@10            9.359632   4.518141
10x_datasets_1-1@100          -3.024895  16.453638
10x_datasets_1-1@101           9.727308  -0.282022
...                                 ...        ...
Wimmers_Cell_2023-76_CC0829@5 -0.019905   0.769589
Wimmers_Cell_2023-76_CC0829@6  8.916915  18.234358
Wimmers_Cell_2023-76_CC0829@7  7.396175   7.055951
Wimmers_Cell_2023-76_CC0829@8 -3.258058   2.601553
Wimmers_Cell_2023-76_CC0829@9 -0.350541  -0.491347

[21795 rows x 2 columns]


In [17]:
metacell_coords.to_csv('./process/umap.csv')

In [18]:
ad = sc.read_h5ad('./process/rna.h5ad')
ad

AnnData object with n_obs × n_vars = 21795 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'

In [19]:
metacell_coords = metacell_coords.reindex(ad.obs_names)
metacell_coords

,UMAP_1,UMAP_2
Cheong_Cell_2023_1-HDu@0,-5.717067,11.605882
Cheong_Cell_2023_1-HDu@1,8.287677,9.587166
Cheong_Cell_2023_1-HDu@2,-1.791123,16.969864
Cheong_Cell_2023_1-HDu@3,14.551053,12.544753
Cheong_Cell_2023_1-HDu@4,6.523927,-0.655004
...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,10.675597,15.611766
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,12.752496,14.712941
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,13.705600,11.237791
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,9.582911,3.703958


In [21]:
ad.obsm['X_umap'] = metacell_coords.to_numpy()

In [ ]:
sc.pl.umap(ad, color='celltype')

In [ ]:
sc.pl.umap(rna,color='l3')

In [27]:
rna_f = rna[rna.obs['l3'].isin(ad.obs['celltype'])]
rna_f

View of AnnData object with n_obs × n_vars = 1640897 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType_raw', 'cellType_raw_level1', 'cellType_raw_level2', 'cellType_raw_level3', 'cellType_scarch_level3', 'cellType_actinn_level3', 'cellType_celltypist_level3', 'cellType_scmap_level3', 'cellType_singlecellnet_level3', 'cellType_scpred_level3', 'cellType_singler_level3', 'cellType_chetah_level3', 'cellType_scarch_level2', 'cellType_actinn_level2', 'cellType_celltypist_level2', 'cellType_scmap_level2', 'cellType_singlecellnet_level2', 'cellType_scpred_level2', 'cellType_singler_level2', 'cellType_chetah_level2', 'tissue', 'disease_name', 'disease', 'age', 'age_stage', 'sex', 'technology', 'final_ann', 'l3', 'SEACell'
    uns: 'l3_colors'
    obsm: 'X_spectral', 'X_umap'
    layers: 'counts'

In [ ]:
sc.pl.umap(ad, color='celltype', palette=rna_f.uns['l3_colors'].tolist(), show=True, save='seacells_cell', size=8)

In [ ]:
sc.pl.umap(ad, color='celltype_purity', show=True, save='_purity')

In [44]:
ad.obs['total'] = 'total'

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 8))  
sc.pl.violin(ad, keys='compactness', groupby='total', rotation=90, ax=ax, save='_compactness_total')
plt.show()

# Contour

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize = (6,6))

ov.pl.embedding(ad,
                basis='X_umap',
                color=['celltype'],
                frameon='small',
                show=False,
                ax=ax)

ov.pl.contour(ax=ax,adata=ad,groupby='celltype',clusters=[
'ELP',
'CMP',
'MEMP',
'HSC/MPP',
'Mid erythroid',
'Late erythroid',
'Early erythroid'
],
       basis='X_umap',contour_threshold=0.1,colors='#000000',
        linestyles='dashed',)

plt.savefig('./figures/contour/hspc_ery.pdf', format='pdf', bbox_inches='tight')

# Draw the corresponding relationship between seacells and original cells

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import omicverse as ov

In [2]:
adata = sc.read_h5ad('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/adata.h5ad')

In [14]:
umap = np.load('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony_umap.npy')
adata.obsm['X_umap'] = umap
ann = pd.read_csv('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/subcluster/celltype_l1_l3.csv', index_col=0)
adata.obs['l3'] = ann.loc[adata.obs_names, 'l3'].values

In [9]:
merged_df = merged_df.reindex(adata.obs_names)
merged_df

,SEACell
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-lane1@12
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-lane1@14
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-lane1@15
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-lane1@52
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-lane1@42
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Marderstein_Nature_2024-H_7_C@23
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Marderstein_Nature_2024-H_7_C@6
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Marderstein_Nature_2024-H_7_C@8
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Marderstein_Nature_2024-H_7_C@32


In [10]:
merged_df.to_csv('./process/cell_assign_all.csv')

In [11]:
adata.obs['SEACell'] = merged_df['SEACell']

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,8))
ov.pl.embedding(
    adata,
    basis="X_umap",
    color=['l3'],
    frameon='small',
    title="Meta cells",
    #legend_loc='on data',
    legend_fontsize=14,
    legend_fontoutline=2,
    size=1,
    ax=ax,
    alpha=0.1,
    #legend_loc='', 
    add_outline=False, 
    #add_outline=True,
    outline_color='black',
    outline_width=1,
    show=False,
    #palette=ov.utils.blue_color[:],
    #legend_fontweight='normal'
)
ov.single.plot_metacells(ax,adata,color='#CB3E35', size=1)

plt.savefig('./figures/all_metacells_plot.pdf', format='pdf', bbox_inches='tight')

# Trajectory analysis-CellRank Meets CytoTRACE

### Import packages & data

In [1]:
import cellrank as cr
import scanpy as sc
import scvelo as scv

cr.settings.verbosity = 2
sc.settings.set_figure_params(frameon=False, dpi=100)

In [2]:
import warnings

warnings.simplefilter("ignore", category=UserWarning)

In [3]:
adata = sc.read('./process/hspc/hspc_raw.h5ad')
adata

AnnData object with n_obs × n_vars = 1353 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'
    uns: 'celltype_colors'
    obsm: 'X_umap'

In [4]:
adata.X.max()

676.1503407161053

### Explore the data

This zebrafish dataset contains 12 time-points spanning 3.3 - 12 hours past fertilization. We can use the original force-directed layout to plot cells, colored by developmental stage:

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color="celltype")

In [ ]:
sc.pl.embedding(adata, basis="X_umap", color="CD34")

### Preprocess

Before we can start applying the CytoTRACE kernel, we'll have to do some basic preprocessing of the data. 

In [7]:
# filter, normalize total counts and log-transform
sc.pp.filter_genes(adata, min_cells=10)
scv.pp.normalize_per_cell(adata, enforce=True)
sc.pp.log1p(adata)

# hvg annotation
sc.pp.highly_variable_genes(adata)
print(f"This detected {adata.var['highly_variable'].sum()} highly variable genes. ")

Normalized count data: X.
This detected 5150 highly variable genes. 


CytoTRACE uses imputed gene expression to calculate the score of differentiation potential. Here, we use use {func}`scvelo.pp.moments` for imputation. We're copying our {attr}`~anndata.AnnData.X` matrix into the layers because that's where {func}`~scvelo.pp.moments` expects to find counts for imputation. 

In [8]:
adata.layers["spliced"] = adata.X
adata.layers["unspliced"] = adata.X
scv.pp.moments(adata, n_pcs=30, n_neighbors=30)

/tmp/ipykernel_2023701/3936592386.py:3: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=30, n_neighbors=30)
/home/qiaolibo/miniconda3/envs/cellrank/lib/python3.10/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
/home/qiaolibo/miniconda3/envs/cellrank/lib/python3.10/site-packages/scvelo/preprocessing/neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute PCA with Scanpy first.
  _set_pca(adata=adata, n_pcs=n_pcs, use_highly_variable=use_highly_variable)


computing neighbors
    finished (0:00:06) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:23) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)


## Use developmental potential to direct differentiation

### Compute the CytoTRACE score

Import the {class}`~cellrank.kernels.CytoTRACEKernel`, initialize it using the preprocessed {class}`~anndata.AnnData` object and compute the CytoTRACE score: 

In [9]:
from cellrank.kernels import CytoTRACEKernel

ctk = CytoTRACEKernel(adata).compute_cytotrace()

Computing CytoTRACE score with `30665` genes
Adding `adata.obs['ct_score']`
       `adata.obs['ct_pseudotime']`
       `adata.obs['ct_num_exp_genes']`
       `adata.var['ct_gene_corr']`
       `adata.var['ct_correlates']`
       `adata.uns['ct_params']`
    Finish (0:00:26)


We can visually check that our computed CytoTRACE pseudotime correlates well with ground-truth real-time annotations:

In [10]:
adata

AnnData object with n_obs × n_vars = 1353 × 30665
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness', 'n_counts', 'ct_score', 'ct_pseudotime', 'ct_num_exp_genes'
    var: 'n_cells', 'gene_count_corr', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ct_gene_corr', 'ct_correlates'
    uns: 'celltype_colors', 'log1p', 'hvg', 'pca', 'neighbors', 'ct_params'
    obsm: 'X_umap', 'X_pca'
    varm: 'PCs'
    layers: 'spliced', 'unspliced', 'Ms', 'Mu'
    obsp: 'distances', 'connectivities'

In [ ]:
sc.pl.embedding(
    adata,
    color=["ct_pseudotime", "celltype"],
    basis="X_umap",
    color_map="gnuplot2",
    # save='cytotrace'
)

To make this a bit more quantitative, we can look at the distribution of our CytoTRACE pseudotime over real time points and show this via violin plots:

In [ ]:
sc.pl.violin(adata, keys=["ct_pseudotime"], groupby="celltype", rotation=90)

As expected, the CytoTRACE pseudotime increases for later time points. Thus, we can use this score to bias graph edges into the direction of increasing differentiation status, similar to how we do this for other pseudotimes with the {class}`~cellrank.kernels.PseudotimeKernel`. 

### Compute & visualize a transition matrix

Compute the transition matrix using {meth}`~cellrank.kernels.CytoTRACEKernel.compute_transition_matrix`. 

In [13]:
ctk.compute_transition_matrix(threshold_scheme="soft", nu=0.5)

Computing transition matrix based on pseudotime


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1353/1353 [00:00<00:00, 4456.00cell/s]


    Finish (0:00:00)


CytoTRACEKernel[n=1353, dnorm=False, scheme='soft', b=10.0, nu=0.5]

To get some intuition for this transition matrix, we can project it into an embedding and draw streamlines - the following plot will look like the plots you're used to from scVelo and velocyto for visualizing RNA velocity {cite}`bergen:20,manno:18`. 

:::{note}

There's **no RNA velocity** here, we're visualizing the directed transition matrix we computed using the kNN graph as well as the CytoTRACE pseudotime.

:::

In [ ]:
ctk.plot_projection(basis="X_umap", color="celltype", legend_loc="right", size=100, 
                    # save='ct'
                   )

Another way to gain intuition for the transition matrix is by drawing some cells from the early stage and to use these as starting cells to simulate random walks on the transition matrix. This visualization technique is less biased by the low-dimensional embedding as the random walks are computed directly based on the transition matrix. 

In [ ]:
ctk.plot_random_walks(
    n_sims=100,
    start_ixs={"celltype": "HSC/MPP"},
    basis="X_umap",
    color="celltype",
    legend_loc="right",
    seed=1,
    # save='cytotrace_walk.pdf'
)

Black dots denote sampled starting cells for random walks, yellow dots denote end cells. We terminated each random walk after a predefined number of steps. Random walks are colored according to how long they've been running for - i.e. the initial segments are more black/blue whereas the late segments are more orange/yellow. We can see that many random walks terminate in one of the two terminal states, as expected. Quite a few of them seem to get stuck early on, this could be further investigated. 

:::{note}

The visualization techniques demonstrated here work for **every kernel**, no matter whether it's a {class}`~cellrank.kernels.VelocityKernel`, {class}`~cellrank.kernels.PseudotimeKernel`, {class}`~cellrank.kernels.CytoTRACEKernel` or some other CellRank {mod}`kernel <cellrank.kernels>`. 

:::

## Closing matters

### What's next? 

In this tutorial, you learned how to use CellRank to compute a transition matrix based on the CytoTRACE pseudotime and how it can be visualized in low dimensions. The real power of CellRank comes in when you use estimators to analyze the transition matrix directly, rather than projecting it. For the next steps, we recommend to:

- go through the {doc}`initial and terminal states tutorial <../estimators/600_initial_terminal>` to learn how to use the transition matrix to automatically identify initial and terminal states. 
- take a look at the {class}`API <cellrank.kernels.CytoTRACEKernel>` to learn about parameter values you can use to adapt these computations to your data. 
- read the original CytoTRACE publication and considering limitations of this appraoch {cite}`gulati:20`. 

### Package versions

In [16]:
cr.logging.print_versions()

cellrank==2.0.7 scanpy==1.10.4 anndata==0.11.4 numpy==1.26.4 numba==0.61.2 scipy==1.14.0 pandas==2.2.3 pygpcca==1.0.4 scikit-learn==1.6.1 statsmodels==0.14.4 scvelo==0.3.3 pygam==0.9.1 matplotlib==3.10.1 seaborn==0.13.2


## gene_trends

In [17]:
import cellrank as cr
import scvelo as scv
import scanpy as sc

In [18]:
adata

AnnData object with n_obs × n_vars = 1353 × 30665
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness', 'n_counts', 'ct_score', 'ct_pseudotime', 'ct_num_exp_genes'
    var: 'n_cells', 'gene_count_corr', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'ct_gene_corr', 'ct_correlates'
    uns: 'celltype_colors', 'log1p', 'hvg', 'pca', 'neighbors', 'ct_params', 'T_fwd_params'
    obsm: 'X_umap', 'X_pca', 'T_fwd_umap'
    varm: 'PCs'
    layers: 'spliced', 'unspliced', 'Ms', 'Mu'
    obsp: 'distances', 'connectivities'

In [19]:
ptk = cr.kernels.PseudotimeKernel(adata, time_key="ct_pseudotime")
ptk.compute_transition_matrix()

estimator = cr.estimators.GPCCA(ptk)
estimator.compute_schur()

estimator.compute_macrostates(3, cluster_key="celltype")
estimator.predict_terminal_states()
estimator.compute_fate_probabilities()

Computing transition matrix based on pseudotime


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1353/1353 [00:00<00:00, 5341.32cell/s]

    Finish (0:00:00)


Computing Schur decomposition
When computing macrostates, choose a number of states NOT in `[18]`
Adding `adata.uns['eigendecomposition_fwd']`
       `.schur_vectors`
       `.schur_matrix`
       `.eigendecomposition`
    Finish (0:00:00)
Computing `3` macrostates
Adding `.macrostates`
       `.macrostates_memberships`
       `.coarse_T`
       `.coarse_initial_distribution
       `.coarse_stationary_distribution`
       `.schur_vectors`
       `.schur_matrix`
       `.eigendecomposition`
    Finish (0:00:00)
Adding `adata.obs['term_states_fwd']`
       `adata.obs['term_states_fwd_probs']`
       `.terminal_states`
       `.terminal_states_probabilities`
       `.terminal_states_memberships
    Finish`
Computing fate probabilities


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 60.80/s]

Adding `adata.obsm['lineages_fwd']`
       `.fate_probabilities`
    Finish (0:00:00)


In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","KLF1","EBF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
#    save='gene_trends.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    # save='gene_trends_heatmap_elp.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    # save='gene_trends_heatmap_MEMP.pdf'
)

## accessible trend

In [8]:
import pandas as pd
import scanpy as sc

In [2]:
arc = pd.read_csv('./process/atac_seacell/GeneAccessibility_all.csv', index_col=0)

In [5]:
arc = arc.where(arc >= 0, 0).fillna(0)

In [6]:
arc

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
Cheong_Cell_2023_1-HDu@0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
adata = sc.AnnData(
    X=arc.values,          
    obs=arc.index.to_frame(name='cell_id'),  
    var=arc.columns.to_frame(name='gene_name')  
)

In [10]:
adata.write_h5ad('./process/arc.h5ad')

In [11]:
arc_hspc = arc.loc[adata.obs_names,adata.var_names]
arc_hspc

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
Cheong_Cell_2023_1-HDu@0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-HDu@4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
arc_hspc = arc_hspc.where(arc_hspc >= 0, 0).fillna(0)

In [28]:
arc_hspc

,MIR1302-2HG,AL627309.1,AL627309.3,AL627309.5,AL627309.4,AP006222.2,AC114498.1,AL669831.2,LINC01409,FAM87B,...,AC011043.2,AC011841.1,AL354822.1,AL592183.1,AC240274.1,AC004556.3,AC233755.2,AC233755.1,AC007325.4,AC007325.2
NeurIPS_2021-site3_donor6@14,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
NeurIPS_2021-site3_donor7@0,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-jcov21@7,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-jcov21@10,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Cheong_Cell_2023_1-jcov21@11,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wei_Immuni_2023-YW4@139,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Wei_Immuni_2023-YW4@152,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Wei_Immuni_2023-YW4@169,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0
Wei_Immuni_2023-YW4@182,0,0,0,0,0,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
adata.X = arc_hspc

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","EBF1","KLF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
    save='gene_trends_atac.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_elp_atac.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_memp_atac.pdf'
)

In [33]:
scv.pp.normalize_per_cell(adata, enforce=True)
sc.pp.log1p(adata)

Normalized count data: X, spliced, unspliced.


In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["GATA1","EBF1","KLF1","DNTT"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
    save='gene_trends_atac_norm.pdf'
)

In [ ]:
model = cr.models.GAM(adata)
cr.pl.gene_trends(
    adata,
    model=model,
    genes=["CD34"],
    time_key="ct_pseudotime",
    hide_cells=True,
    same_plot=True,
#    save='gene_trends.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","EBF1","DNTT"],
    lineages=["ELP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_elp_atac_norm.pdf'
)

In [ ]:
cr.pl.heatmap(
    adata,
    model=model,
    genes=["CD34","GATA1","KLF1"],
    lineages=["MEMP"],
    time_key="ct_pseudotime",
    cbar=False,
    show_all_genes=True,
    save='gene_trends_heatmap_memep_atac_norm.pdf'
)

# Chromatin potential

## Gene accessibility scores

This notebook is a tutorial for ATAC analysis using SEACells and includes computation of gene-peak associations, ATAC gene scores, gene accessibility scores and identification of highly regulated genes

### Imports

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
import SEACells

/home/qiaolibo/miniconda3/envs/seacells/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Some plotting aesthetics
%matplotlib inline

sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100


### Load Data

In [5]:
rna_meta_ad = sc.read('process/rna.h5ad')
atac_meta_ad = sc.read('data/atac_meta_ad.h5ad')

In [7]:
# # Revise var_names
# def replace_first_dash(s):
#     return s.replace('-', ':', 1)

# atac_meta_ad.var_names = [replace_first_dash(name) for name in atac_meta_ad.var_names]

# # verify
# print(atac_meta_ad.var_names[:5])

Index(['chr1:792623-793123', 'chr1:794173-794673', 'chr1:804657-805157',
       'chr1:807626-808126', 'chr1:811266-811766'],
      dtype='object')


In [8]:
# def _add_atac_meta_data(atac_meta_ad, atac_ad, n_bins_for_gc):
#     atac_ad.var["log_n_counts"] = np.ravel(np.log10(atac_ad.X.sum(axis=0)))

#     atac_meta_ad.var["GC_bin"] = np.digitize(
#         atac_ad.var["GC"], np.linspace(0, 1, n_bins_for_gc)
#     )
#     atac_meta_ad.var["counts_bin"] = np.digitize(
#         atac_ad.var["log_n_counts"],
#         np.linspace(
#             atac_ad.var["log_n_counts"].min(),
#             atac_ad.var["log_n_counts"].max(),
#             n_bins_for_gc,
#         ),
#     )

####  add gc_bin

In [9]:
# import pandas as pd
# from pyfaidx import Fasta
# from Bio.SeqUtils import gc_fraction as GC

In [10]:
# def calculate_gc_content(bed_file: str, genome_fasta: str) -> pd.DataFrame:
#     """
#     Calculates the GC content for each region in a BED file and returns a DataFrame with detailed results.
    
#     Args:
#         bed_file (str): Path to the BED file (format: chr start end [optional columns]).
#         genome_fasta (str): Path to the reference genome FASTA file (must be indexed, e.g., with samtools).
    
#     Returns:
#         pd.DataFrame: A table containing chromosome, start, end, sequence, and GC content.
    
#     Example:
#         >>> df = calculate_gc_content("peaks.bed", "hg38.fa")
#         >>> df.head()
#     """
#     try:
#         # Load reference genome
#         genome = Fasta(genome_fasta)
        
#         # Read BED file as DataFrame (automatically handles comment lines and column names)
#         bed_df = pd.read_csv(
#             bed_file, 
#             sep='\t', 
#             comment='#', 
#             header=None,
#             names=['chr', 'start', 'end'] + [f'extra_{i}' for i in range(3, 100)]  # Handle potential extra columns
#         )
        
#         # Initialize results list
#         results = []
        
#         for _, row in bed_df.iterrows():
#             chrom, start, end = row['chr'], int(row['start']), int(row['end'])
            
#             try:
#                 # Extract sequence (BED is 0-based, pyfaidx uses 1-based indexing)
#                 sequence = str(genome[chrom][start+1:end].seq).upper()  # Ensure uppercase
                
#                 # Calculate GC content
#                 gc_percent = GC(sequence)
                
#                 # Save results (including original BED rows and calculated results)
#                 result_row = row.to_dict()
#                 result_row.update({
#                     'sequence': sequence,
#                     'gc_percent': gc_percent,
#                     'length': len(sequence)
#                 })
#                 results.append(result_row)
                
#             except (KeyError, IndexError) as e:
#                 print(f"Warning: Skipping invalid region {chrom}:{start}-{end} (Error: {str(e)})")
#                 continue
                
#         # Convert to DataFrame
#         result_df = pd.DataFrame(results)
        
#         # Reorder columns (place key information at the beginning)
#         base_cols = ['chr', 'start', 'end', 'length', 'gc_percent', 'sequence']
#         other_cols = [col for col in result_df.columns if col not in base_cols]
#         return result_df[base_cols + other_cols]
    
#     except Exception as e:
#         raise RuntimeError(f"Failed to calculate GC content: {str(e)}") from e

In [11]:
# results = calculate_gc_content('/pool1/qiaolibo/R_Project/peak_gene/z_others/0_ref_peak/merged_summit/All_Samples.fwp.filter.non_overlapping.bed', 
#                                '/pool1/qiaolibo/ref/refdata-gex-GRCh38-2020-A/fasta/genome.fa')

In [12]:
# results.iloc[:,:8].to_csv('./data/peak_gc.csv')

In [13]:
# result = pd.read_csv('./data/peak_gc.csv', index_col=0)
# result

,chr,start,end,length,gc_percent,sequence,extra_3,extra_4
0,chr1,792622,793123,500,0.420,AAAAGTTTGTGTTCTCTCTGTAGTTCTTCTGAAATTTAGAAATGTT...,Oguchi_Science_2024_multiome-snATACseq_D1_Bulk...,1.557763
1,chr1,794172,794673,500,0.522,ATTTCCCACAATTTCCTTAGAGAATATCCATGCAGTGATGGTCACC...,Oguchi_Science_2024-multiome-snATACseq_D2_Th17...,1.646770
2,chr1,804656,805157,500,0.410,GTAGACAGATGGTCAAGAATTGCTACTATAAAGCAACTCAGTATGG...,Mimitou_Nat_Biotechnol_2021_2023_GSM5065528_LL...,3.795381
3,chr1,807625,808126,500,0.406,TCTGATTCATTCTATACCCACTAACCATAACCAAACAGATTAAGTT...,Weng_Nature_2024_GSE219248_Young2_1,1.238993
4,chr1,811265,811766,500,0.462,TTTGTATTTTATATTTAGTCTTACACCAGTTTGTTATCATTGATTG...,Oguchi_Science_2024-multiome-snATACseq_D2_Th1o...,3.659859
...,...,...,...,...,...,...,...,...
527454,chrX,155966771,155967272,500,0.414,CCAAATCCAAGGTTGCACAGGCTTTCTTCTAGAATGTTTATAGTTT...,Thomson_Nat_Immunol_2024_GSM6611366_B076-P1_PB...,15.058664
527455,chrX,155968030,155968531,500,0.464,TGTCTCATGACCAAGAAAGTTAAGGAGTGCAGACACAAAGGGTGAG...,Cheong_Cell_2023_1_lgtd18_113897,14.442810
527456,chrX,155968729,155969230,500,0.478,GGCTTTTCATGGTCTACTGGCCCATGAGTGCTCCTGCCTGCTCCCC...,Li_Nat_Biotechnol_2023_GSM6403411_3423-MV-2_18...,4.356075
527457,chrX,155971565,155972066,500,0.464,CAGGGTTTGTCTGGGCTCGGTTTTGCCTCCCAGGTTTCAGTGGCTG...,Weng_Nature_2024_GSE219248_Young2_187195,2.371047


In [14]:
# atac_meta_ad.var['GC'] = result['gc_percent'].tolist()

In [15]:
# atac_meta_ad.var["GC_bin"] = np.digitize(
#     atac_meta_ad.var["GC"], np.linspace(0, 1, 50)
# )

In [16]:
# atac_meta_ad.var["GC_bin"]

chr1:792623-793123          21
chr1:794173-794673          26
chr1:804657-805157          21
chr1:807626-808126          20
chr1:811266-811766          23
                            ..
chrX:155966772-155967272    21
chrX:155968031-155968531    23
chrX:155968730-155969230    24
chrX:155971566-155972066    23
chrX:155975921-155976421    26
Name: GC_bin, Length: 527459, dtype: int64

#### add counts_bin

In [17]:
# sc.pp.calculate_qc_metrics(atac_meta_ad,inplace=True)

In [19]:
# atac_meta_ad.var["counts_bin"] = np.digitize(
#         atac_meta_ad.var["log1p_total_counts"],
#         np.linspace(
#             atac_meta_ad.var["log1p_total_counts"].min(),
#             atac_meta_ad.var["log1p_total_counts"].max(),
#             50,
#         ),
# )

In [20]:
# atac_meta_ad.var["counts_bin"]

chr1:792623-793123          12
chr1:794173-794673          13
chr1:804657-805157          12
chr1:807626-808126          12
chr1:811266-811766          12
                            ..
chrX:155966772-155967272    26
chrX:155968031-155968531    21
chrX:155968730-155969230    16
chrX:155971566-155972066    16
chrX:155975921-155976421    15
Name: counts_bin, Length: 527459, dtype: int64

In [22]:
# atac_meta_ad.write_h5ad('./data/atac_meta_ad.h5ad')

### Gene-peak correlations

Using the paired multiome metacell data, the next step is to compute the correlation of gene expression and accessbility of peaks within the vicinity of the gene. 

Computation of gene peak correlations requires the following parameters :
1. GTF file with gene annotations. <b> Note: </b> Chromosome names should be numbered 1, 2 and the "chr" prefix will be added by SEACells 
2. Genomic span around genes to test the correlations 

Human GTF file is available at [https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf](https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf) and can be downloaded using:

In [8]:
#! wget https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/hg38.gtf -O data/hg38.gtf

In [7]:
# In this example, we compute gene peak correlations for the first ten genes since 
# this process is computationally intensive
# gene_set = rna_meta_ad.var_names
# gene_peak_cors = SEACells.genescores.get_gene_peak_correlations(atac_meta_ad, rna_meta_ad, 
#                                            path_to_gtf='data/hg38.gtf', 
#                                            span=100000, 
#                                            n_jobs=100,
#                                            gene_set=gene_set)

In [ ]:
# import pickle

# with open('gene_peak_cors.pkl', 'wb') as f:
#     pickle.dump(gene_peak_cors, f)

#### Highly regulated genes

In [7]:
gene_peak_cors = pd.read_pickle('./atac_seacell/all/gene_peak_cors.pkl') 

In [18]:
len(gene_peak_cors)

36601

Highly regulated genes i.e., genes that are correlated with multiple peaks can be identified using the `get_gene_peak_assocations` function. `get_gene_peak_assocations` returns the number of significantly peaks correlated with each gene

In [19]:
peak_counts = SEACells.genescores.get_gene_peak_assocations(gene_peak_cors, 
                                                           pval_cutoff=1e-1,
                                                           cor_cutoff=0.1)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36601/36601 [00:04<00:00, 8709.89it/s]


In [11]:
peak_counts.value_counts()

0      21304
1       1563
2       1472
3       1288
4       1130
       ...  
96         1
130        1
145        1
121        1
95         1
Name: count, Length: 129, dtype: int64

In [ ]:
# Plot the distribution to identify genes with higher degree of regulation
plt.scatter(np.arange(len(peak_counts)), 
           np.sort(peak_counts), s=20)
sns.despine()
plt.xlabel('Gene rank')
plt.ylabel('No. of correlated peaks')

#### Gene scores

Gene scores are computed as the weighted sum of the accessiblity of correlated peaks and can be computed using `get_gene_scores`. 

In [ ]:
# gene_scores = SEACells.genescores.get_gene_scores(atac_meta_ad, 
#                                                   gene_peak_cors,
#                                                   pval_cutoff=1e-1,
#                                                   cor_cutoff=0.1)

  1%|█▍                                                                                                               | 485/36601 [17:36<28:36:11,  2.85s/it]

In [ ]:
gene_scores = pd.read_csv('gene_scores.csv')

In [28]:
gene_scores.head()

,LINC01128,FAM41C,NOC2L,ISG15,AGRN,C1orf159,SDF4,B3GALT6,UBE2J2,SCNN1D,...,MPP1,F8,FUNDC2,CMC4,MTCP1,BRCC3,VBP1,RAB39B,CLIC2,TMLHE
cd34_multiome_rep1#TTGCAGCCAGGCGATA-1,0.0,0.005829,0.009785,0.018159,0.053374,0.042345,0.036484,0.014096,0.029941,0.003039,...,0.006971,0.002203,0.000000,0.001719,0.015165,0.004225,0.00000,0.035308,0.034425,0.005338
cd34_multiome_rep1#TACAGCTAGTAATCCA-1,0.0,0.017462,0.007034,0.021173,0.026875,0.018844,0.034083,0.020050,0.033753,0.012982,...,0.024762,0.007021,0.000000,0.001290,0.002502,0.007835,0.00000,0.051158,0.050683,0.004603
cd34_multiome_rep2#TAACCTAAGGATTGAG-1,0.0,0.026632,0.008566,0.027379,0.026118,0.020141,0.030814,0.012761,0.033828,0.013474,...,0.012807,0.000000,0.000000,0.000000,0.011502,0.001524,0.00000,0.045826,0.043820,0.003887
cd34_multiome_rep1#TAGCGGCTCATTATGG-1,0.0,0.003788,0.024816,0.008464,0.016567,0.020965,0.078593,0.065220,0.093520,0.049712,...,0.124604,0.036083,0.003839,0.016096,0.007937,0.003655,0.03626,0.057396,0.059264,0.007541
cd34_multiome_rep1#GCCCATAAGATGCCTG-1,0.0,0.021467,0.005354,0.019011,0.016332,0.017859,0.024918,0.019455,0.024968,0.008479,...,0.017892,0.004407,0.000000,0.002284,0.010087,0.003307,0.00000,0.055975,0.082281,0.006632


In [29]:
gene_scores.shape

(99, 12464)

`gene_scores` is a `pandas.DataFrame` with metacells as rows and genes as columns. This can be used for any downstream analysis such as clustering, visualization etc.

### Gene-accessibility

This section describes how to compute gene accessiblity metrics using SEACell metacells.

#### Open peaks in metacells

The first step is to identify the subset of peaks that are open in each metacell. `determine_metacell_open_peaks` function can be used to determine this. 

By default, all peaks are tested to check if they are open or closed in every metacell. A subset of peaks can be specified using the `peak_set` parameter. 

This function also requires a low-dimensional embedding such as `X_svd`. We can summarize the SVD of single-cell ATAC for this analysis

In [8]:
atac_meta_ad

AnnData object with n_obs × n_vars = 21795 × 527459
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'GC', 'GC_bin', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'counts_bin'

In [9]:
atac_meta_ad.obs['n_counts'] = atac_meta_ad.obs['total_counts']

In [10]:
# read spectral

In [11]:
spectral = np.load('/pool1/qiaolibo/R_Project/peak_gene/z_others/8_new/processed/integrate/multi_spectral/multi_spectral_harmony.npy')
spectral

array([[-7.21022992e-04, -2.46545940e-05, -1.13143921e-04, ...,
        -1.02716423e-04,  1.94733056e-05,  8.66159265e-05],
       [-7.17287217e-04,  5.93032215e-06, -1.88112696e-04, ...,
         5.34673160e-07,  4.29835585e-05,  3.12961930e-05],
       [-6.78960863e-04,  5.05488933e-05, -5.84620628e-05, ...,
         9.15538062e-05,  9.17580283e-05, -2.47395818e-05],
       ...,
       [-7.69034294e-04, -1.41434810e-05,  4.99319742e-05, ...,
        -4.10238440e-05, -4.08265529e-05, -2.67895688e-05],
       [-7.59690104e-04, -8.03503573e-06, -7.95369510e-05, ...,
        -5.69030152e-05, -2.62458301e-05,  5.75557733e-05],
       [-9.87588949e-04, -3.30815835e-05,  1.76416610e-04, ...,
         1.42418801e-04,  1.54202953e-04,  2.68111141e-06]])

In [12]:
cell_assign = pd.read_csv('./process/cell_assign_all.csv', index_col=0)
cell_assign

,SEACell
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022-lane1@12
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022-lane1@14
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022-lane1@15
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022-lane1@52
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022-lane1@42
...,...
Marderstein_Nature_2024-TTTGTGTTCACGAATC-H_7_C,Marderstein_Nature_2024-H_7_C@23
Marderstein_Nature_2024-TTTGTGTTCATGGCCA-H_7_C,Marderstein_Nature_2024-H_7_C@6
Marderstein_Nature_2024-TTTGTGTTCTGTGCAG-H_7_C,Marderstein_Nature_2024-H_7_C@8
Marderstein_Nature_2024-TTTGTTGGTAGGTTTG-H_7_C,Marderstein_Nature_2024-H_7_C@32


In [13]:
sc_spectral = pd.DataFrame(spectral, index=cell_assign.index)

In [14]:
atac_meta_ad.obsm['X_spectral'] = sc_spectral.groupby(cell_assign['SEACell']).mean().loc[atac_meta_ad.obs_names, :]

In [15]:
atac_meta_ad

AnnData object with n_obs × n_vars = 21795 × 527459
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'n_counts'
    var: 'GC', 'GC_bin', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'counts_bin'
    obsm: 'X_spectral'

In [33]:
# # We will reuse the atac_meta_ad computed above

# # Add SVD summary to atac meta ad
# seacell_label = 'SEACell'
# sc_svd = pd.DataFrame(atac_ad.obsm['X_svd'], index=atac_ad.obs_names)
# atac_meta_ad.obsm['X_svd'] = sc_svd.groupby(atac_ad.obs[seacell_label]).mean().loc[atac_meta_ad.obs_names, :]

In [13]:
# # Determine open peaks in each metacell
# SEACells.accessibility.determine_metacell_open_peaks(atac_meta_ad, peak_set=None, low_dim_embedding='X_spectral', pval_cutoff=1e-2,
#                                   read_len=147, n_neighbors=3, n_jobs=1)
# # This function will add 'OpenPeaks' to the Anndata layers and is a binary matrix 
# # indicating whether the peak is open or closed in the metacell

In [52]:
# np.save('OpenPeaks.npy', atac_meta_ad.layers['OpenPeaks'])

In [16]:
atac_meta_ad.layers['OpenPeaks'] = np.load('./atac_seacell/all/OpenPeaks.npy')

#### Gene accessibility metric

Open peaks are used to compute gene accessiblity metric which represents the fraction of correlated open peaks. 

<b>Warning: </b> This metric is only reliable if there are sufficient number of open peaks associated with each gene. 
It is recommended to be used for only genes with high regulation

In [ ]:
# Use the highly regulated genes as the gene set of interest 
# Plot the distribution to identify genes with higher degree of regulation
plt.scatter(np.arange(len(peak_counts)), 
           np.sort(peak_counts), s=20)
sns.despine()
plt.xlabel('Gene rank')
plt.ylabel('No. of correlated peaks')

In [40]:
# Select genes based on the elbow point
high_reg_genes = peak_counts.index[peak_counts > 9]

In [47]:
high_reg_genes

Index(['C1orf159', 'LINC01342', 'TTLL10', 'TNFRSF18', 'TNFRSF4', 'MXRA8',
       'ATAD3C', 'ATAD3B', 'SKI', 'PANK4',
       ...
       'FGF13', 'SLITRK4', 'AFF2', 'IDS', 'CXorf40A', 'PDZD4', 'TKTL1', 'FLNA',
       'EMD', 'GAB3'],
      dtype='object', length=6030)

In [42]:
gene_peak_cors

MIR1302-2HG    0
FAM138A        0
OR4F5          0
AL627309.1     0
AL627309.3     0
              ..
AC141272.1     0
AC023491.2     0
AC007325.1     0
AC007325.4     0
AC007325.2     0
Length: 36601, dtype: object

In [ ]:
# # Compute gene accessibility
SEACells.accessibility.get_gene_accessibility(atac_meta_ad, gene_peak_cors, 
                                              gene_set=peak_counts.index, pval_cutoff=1e-1, cor_cutoff=0.1)
# p-value and corrrelation cutoffs are used for correlated peaks
# # This function will add 'GeneAccessibility' to the Anndata `.obsm` field

In [21]:
atac_meta_ad.obsm['GeneAccessibility']

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
Cheong_Cell_2023_1-HDu@0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@2,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@3,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@4,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [22]:
atac_meta_ad.obsm['GeneAccessibility'].max()

MIR1302-2HG   -1.0
FAM138A       -1.0
OR4F5         -1.0
AL627309.1    -1.0
AL627309.3    -1.0
              ... 
AC141272.1    -1.0
AC023491.2    -1.0
AC007325.1    -1.0
AC007325.4    -1.0
AC007325.2    -1.0
Length: 36601, dtype: float64

In [ ]:
atac_meta_ad.obsm['GeneAccessibility'].to_csv('./GeneAccessibility_all.csv')

Gene accessiblity metrics can be used as inputs for downstream analyses.

#### Visualization

In [24]:
rna_meta_ad.obs

,Pseudo-sizes,celltype,celltype_purity,separation,compactness
Cheong_Cell_2023_1-HDu@0,63.736985,Tem/Trm cytotoxic T cells,0.480005,23.204641,0.009738
Cheong_Cell_2023_1-HDu@1,65.422619,Tcm/Naive helper T cells,0.468285,12.636931,30.667777
Cheong_Cell_2023_1-HDu@2,77.541340,Naive B cells,0.503537,0.101567,0.010034
Cheong_Cell_2023_1-HDu@3,72.350554,Tcm/Naive helper T cells,0.960469,0.101567,0.018626
Cheong_Cell_2023_1-HDu@4,64.618502,CD16+ NK cells,0.830080,1.198503,0.009845
...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,114.700387,Tcm/Naive helper T cells,0.994548,0.045278,0.012154
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,110.842287,Tcm/Naive helper T cells,1.000000,0.046566,0.002396
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,53.609057,Regulatory T cells,0.855926,0.116286,0.034826
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,50.190145,Type 1 helper T cells,0.981109,0.058782,0.016925


In [25]:
umap = pd.read_csv('./process/umap.csv',index_col=0)
umap = umap.reindex(rna_meta_ad.obs_names)
umap

,UMAP_1,UMAP_2
Cheong_Cell_2023_1-HDu@0,-5.717067,11.605882
Cheong_Cell_2023_1-HDu@1,8.287677,9.587166
Cheong_Cell_2023_1-HDu@2,-1.791123,16.969864
Cheong_Cell_2023_1-HDu@3,14.551053,12.544753
Cheong_Cell_2023_1-HDu@4,6.523927,-0.655004
...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,10.675597,15.611766
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,12.752496,14.712941
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,13.705600,11.237791
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,9.582911,3.703958


In [26]:
rna_meta_ad.obsm['X_umap'] = umap.to_numpy()

In [ ]:
sc.pl.umap(rna_meta_ad, color='celltype')

In [28]:
GeneAccessibility = pd.read_csv('./GeneAccessibility_all.csv', index_col=0)
GeneAccessibility

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
Cheong_Cell_2023_1-HDu@0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@2,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@3,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Cheong_Cell_2023_1-HDu@4,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [29]:
genes = ['KLF1', 'GATA1', 'SPI1']

In [ ]:
sc.pl.scatter(rna_meta_ad, basis='umap', color=genes)

In [31]:
temp = rna_meta_ad[:, genes]

In [32]:
temp

View of AnnData object with n_obs × n_vars = 21795 × 3
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'
    uns: 'celltype_colors'
    obsm: 'X_umap'

In [34]:
temp.obs[genes] = GeneAccessibility[genes]

/tmp/ipykernel_696823/4102362011.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  temp.obs[genes] = GeneAccessibility[genes]


In [35]:
temp.obs

,Pseudo-sizes,celltype,celltype_purity,separation,compactness,KLF1,GATA1,SPI1
Cheong_Cell_2023_1-HDu@0,63.736985,Tem/Trm cytotoxic T cells,0.480005,23.204641,0.009738,0.50,0.0,0.6
Cheong_Cell_2023_1-HDu@1,65.422619,Tcm/Naive helper T cells,0.468285,12.636931,30.667777,0.25,0.0,0.3
Cheong_Cell_2023_1-HDu@2,77.541340,Naive B cells,0.503537,0.101567,0.010034,0.50,0.0,0.5
Cheong_Cell_2023_1-HDu@3,72.350554,Tcm/Naive helper T cells,0.960469,0.101567,0.018626,0.25,0.0,0.2
Cheong_Cell_2023_1-HDu@4,64.618502,CD16+ NK cells,0.830080,1.198503,0.009845,0.25,0.0,0.2
...,...,...,...,...,...,...,...,...
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@401,114.700387,Tcm/Naive helper T cells,0.994548,0.045278,0.012154,0.25,0.0,0.1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@402,110.842287,Tcm/Naive helper T cells,1.000000,0.046566,0.002396,0.25,0.0,0.1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@403,53.609057,Regulatory T cells,0.855926,0.116286,0.034826,0.25,0.0,0.1
Thomson_Nat_Immunol_2024-B076_P1_PB00353_03@404,50.190145,Type 1 helper T cells,0.981109,0.058782,0.016925,0.25,0.0,0.2


In [ ]:
sc.pl.scatter(temp, basis='umap', color=genes)

## Potential

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.spatial.distance import cdist

In [2]:
adata = sc.read_h5ad('./process/chrom_pote.h5ad')
adata

AnnData object with n_obs × n_vars = 1353 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'
    uns: 'celltype_colors'
    obsm: 'X_umap', 'dorc_scores', 'gene_expression'

In [6]:
adata.obsm['dorc_scores']

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [7]:
adata.obsm['gene_expression']

array([[0.        , 0.        , 0.        , ..., 0.        , 0.00566788,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.01263028,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [74]:
spectral = pd.read_csv('./process/spectral.csv', index_col=0).loc[adata.obs_names,:]
spectral

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
NeurIPS_2021-site3_donor6@14,-0.000770,-0.000313,-0.000583,0.000056,0.000220,-0.000466,0.000080,-0.000039,-0.000057,0.000143,...,0.000133,-0.000079,-0.000026,0.000025,0.000064,-0.000021,0.000002,-0.000055,-0.000018,0.000017
NeurIPS_2021-site3_donor7@0,-0.000771,-0.000272,-0.000473,0.000123,0.000172,-0.000234,0.000049,0.000001,0.000011,-0.000016,...,0.000012,0.000009,-0.000061,-0.000019,0.000016,0.000026,-0.000017,-0.000028,-0.000017,-0.000006
Cheong_Cell_2023_1-jcov21@7,-0.000730,-0.000220,-0.000464,0.000047,0.000165,-0.000300,0.000104,0.000012,-0.000111,0.000206,...,0.000200,-0.000081,0.000017,0.000046,0.000091,-0.000028,0.000010,-0.000003,-0.000009,-0.000046
Cheong_Cell_2023_1-jcov21@10,-0.000785,-0.000201,-0.000422,0.000067,0.000152,-0.000190,0.000041,-0.000069,-0.000046,0.000046,...,-0.000052,0.000042,0.000010,-0.000054,-0.000003,0.000066,0.000005,0.000024,-0.000021,-0.000007
Cheong_Cell_2023_1-jcov21@11,-0.000764,-0.000167,-0.000384,0.000032,0.000150,-0.000221,0.000051,-0.000085,-0.000101,0.000099,...,0.000125,-0.000030,-0.000003,0.000003,0.000078,0.000031,-0.000003,-0.000007,-0.000025,-0.000032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wei_Immuni_2023-YW4@139,-0.000747,-0.000619,-0.000473,0.000028,0.000100,-0.000399,-0.000071,-0.000131,-0.000006,0.000235,...,0.000137,-0.000064,0.000006,0.000037,0.000024,-0.000008,0.000012,-0.000038,0.000006,-0.000075
Wei_Immuni_2023-YW4@152,-0.000772,-0.000652,-0.000574,0.000152,0.000144,-0.000391,-0.000045,-0.000228,0.000062,0.000215,...,0.000030,-0.000032,0.000059,-0.000007,-0.000053,-0.000008,0.000066,0.000089,0.000031,0.000063
Wei_Immuni_2023-YW4@169,-0.000685,-0.000763,-0.000505,0.000182,0.000163,-0.000362,-0.000085,-0.000066,0.000107,0.000169,...,-0.000102,0.000005,0.000081,0.000008,-0.000114,-0.000021,0.000066,0.000086,0.000052,0.000057
Wei_Immuni_2023-YW4@182,-0.000782,-0.000816,-0.000244,0.000126,0.000146,-0.000284,-0.000091,-0.000081,0.000033,0.000120,...,-0.000028,0.000006,-0.000053,-0.000085,-0.000008,0.000139,-0.000054,-0.000108,-0.000060,-0.000092


In [75]:
adata.obsm['X_cisTopic'] = spectral.to_numpy()

In [76]:
adata.layers['RNA'] = adata.X

In [77]:
adata.obsm['dorc_scores'] = adata.obsm['dorc_scores']

In [ ]:
sc.pl.umap(adata, color='celltype')

In [79]:
adata

AnnData object with n_obs × n_vars = 1353 × 36601
    obs: 'Pseudo-sizes', 'celltype', 'celltype_purity', 'separation', 'compactness'
    uns: 'celltype_colors'
    obsm: 'X_umap', 'dorc_scores', 'gene_expression', 'X_cisTopic'
    layers: 'RNA'

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import identity
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

def compute_chromatin_potential(adata, 
                               k_smooth=50, 
                               k_neighbors=10,
                               smooth_nn=15,
                               n_grid=40):
    """Main computation function: Chromatin potential calculation workflow."""
    # 1. Data Smoothing
    sc.pp.neighbors(adata, n_neighbors=k_smooth, use_rep='X_cisTopic')
    adj_matrix = adata.obsp['connectivities'].copy()
    adj_matrix += identity(adj_matrix.shape[0])
    row_sum = adj_matrix.sum(axis=1).A1
    norm_adj = adj_matrix / row_sum[:, None]
    
    # Smooth DORC and gene expression
    adata.obsm['dorc_smoothed'] = norm_adj.dot(adata.obsm['dorc_scores'])
    adata.obsm['gene_smoothed'] = norm_adj.dot(adata.obsm['gene_expression'])
    
    # 2. Find RNA Neighbors
    scaler = StandardScaler()
    dorc_scaled = scaler.fit_transform(adata.obsm['dorc_smoothed'])
    gene_scaled = scaler.fit_transform(adata.obsm['gene_smoothed'])
    corr_matrix = np.dot(dorc_scaled, gene_scaled.T) / (dorc_scaled.shape[1]-1)
    top10_indices = np.argsort(-corr_matrix, axis=1)[:, :k_neighbors]
    
    # 3. Calculate Direction Vectors (in UMAP space)
    X_umap = adata.obsm['X_umap']
    target_positions = np.array([X_umap[indices].mean(axis=0) for indices in top10_indices])
    directions = target_positions - X_umap
    distances = np.linalg.norm(directions, axis=1)
    
    # Normalize and store
    adata.obs['chromatin_potential'] = distances / distances.max()
    adata.obsm['raw_directions'] = directions
    
    # 4. Direction Smoothing
    sc.pp.neighbors(adata, n_neighbors=smooth_nn, use_rep='X_umap')
    adj_smooth = adata.obsp['connectivities']
    row_sum_smooth = adj_smooth.sum(axis=1).A1
    norm_adj_smooth = adj_smooth / row_sum_smooth[:, None]
    adata.obsm['smoothed_directions'] = norm_adj_smooth.dot(directions)
    
    # 5. Generate Valid Grid
    generate_valid_grid(adata, n_grid)
    
def generate_valid_grid(adata, n_grid=40):
    """Generate a grid system with valid direction filtering."""
    umap_coords = adata.obsm['X_umap']
    
    # Create grid
    x_min, x_max = umap_coords[:,0].min()-0.1, umap_coords[:,0].max()+0.1
    y_min, y_max = umap_coords[:,1].min()-0.1, umap_coords[:,1].max()+0.1
    grid_x = np.linspace(x_min, x_max, n_grid)
    grid_y = np.linspace(y_min, y_max, n_grid)
    grid_points = np.array(np.meshgrid(grid_x, grid_y)).T.reshape(-1,2)
    
    # Mark valid grid points (grid points containing cells)
    nn = NearestNeighbors(n_neighbors=1).fit(grid_points)
    _, indices = nn.kneighbors(umap_coords)
    grid_counts = np.bincount(indices.flatten(), minlength=len(grid_points))
    valid_mask = grid_counts > 0
    
    # Calculate directions for valid grid points
    valid_directions = []
    for i in np.where(valid_mask)[0]:
        cell_mask = (indices.flatten() == i)
        avg_dir = adata.obsm['smoothed_directions'][cell_mask].mean(axis=0)
        valid_directions.append(avg_dir)
    
    adata.uns['grid_velocity'] = {
        'grid_points': grid_points[valid_mask],
        'directions': np.array(valid_directions)
    }

def plot_chromatin_potential(adata, 
                            scale_factor=50, 
                            grid_scale=80,
                            plot_cells=True,
                            plot_grid=True,
                            save=True,
                            save_path="chromatin_potential.pdf",
                            dpi=300):
    """Visualize chromatin potential and support PDF export."""
    import os
    
    plt.figure(figsize=(10,6), dpi=dpi)
    
    # Plot cell type background
    sc.pl.embedding(adata, basis='X_umap', color='celltype', 
                   alpha=0.3, show=False, frameon=False, title='')
    
    # # Plot cell-level directions
    # if plot_cells:
    #     plt.quiver(adata.obsm['X_umap'][:,0], adata.obsm['X_umap'][:,1],
    #               adata.obsm['smoothed_directions'][:,0],
    #               adata.obsm['smoothed_directions'][:,1],
    #               scale=scale_factor, width=0.002, 
    #               color='grey', alpha=0.5, headlength=3)
    
    # Plot grid-level directions
    if plot_grid and 'grid_velocity' in adata.uns:
        gp = adata.uns['grid_velocity']['grid_points']
        gd = adata.uns['grid_velocity']['directions']
        plt.quiver(gp[:,0], gp[:,1], gd[:,0], gd[:,1],
                  scale=grid_scale, width=0.004, 
                  color='black', headlength=5, headaxislength=4)
    
    plt.title(f"Chromatin Potential (Scale: {scale_factor}/{grid_scale})")
    plt.tight_layout()
    
    # Save PDF
    if save:
        plt.savefig(save_path, bbox_inches='tight', dpi=dpi)
        print(f"PDF saved to: {os.path.abspath(save_path)}")
    
    plt.show()

# Execute complete workflow
compute_chromatin_potential(adata, n_grid=40)
plot_chromatin_potential(adata, scale_factor=60, grid_scale=100)